# Book RAG v2: improved pipeline + realistic evaluation

This notebook upgrades the RAG system from [RAG_Book_Based_LLM.ipynb](RAG_Book_Based_LLM.ipynb) (v1) and evaluates **both** versions on the same held-out test set, so every improvement is measured rather than assumed.

**Pipeline changes** (all in [rag_pipeline.py](rag_pipeline.py), each one a config flag):

| v1 | v2 |
|---|---|
| `HuggingFacePipeline` returned the prompt + context as part of the answer | Mistral chat template, only new tokens decoded, greedy decoding |
| MiniLM dense search only | BGE-small dense + BM25, fused with Reciprocal Rank Fusion |
| No re-ranking | Cross-encoder re-ranker (`BAAI/bge-reranker-base`) |
| Chunks don't know which book they're from | Each chunk is indexed with a `Book: <title> by <author>` header |
| Searches all books for every question | If the question names a book, search only that book |
| Hand-set L2 threshold + "question must contain what/who/how" rule | One abstention threshold on the re-ranker score, **calibrated on a dev split** |
| No sources | Numbered excerpts and `[n]` citations |

**Evaluation** (in [rag_evaluation.py](rag_evaluation.py)):
- **Test set:** 23 hand-checked questions with exact evidence quotes, ~60 synthetic questions spread over all 20 books, and 21 negatives (off-topic, books not in the library, false premises, gibberish). The split is stratified: thresholds are tuned on **dev** and results are reported on **test**.
- **Retrieval:** Hit@k, MRR, and nDCG@10 at the passage level (a hit means the retrieved chunk actually contains the gold evidence), plus how well the abstention score separates answerable from unanswerable questions (AUROC). Run as an ablation over 7 configurations.
- **End to end:** answer accuracy (LLM judge), token F1, faithfulness and hallucination rate (from claim-by-claim checks), citation precision, how often the system refuses answerable questions, and how often it correctly handles unanswerable ones. Also reported: latency p50/p95, 95% bootstrap confidence intervals, and a failure label for each question (retrieval miss / hallucination / generation error).

**Runtime:** about 45-60 minutes on a Colab L4 (most of it is the LLM judge). `MAX_TEST_ITEMS` below gives a faster run.

## 0) Setup

In [ ]:
!pip install -q -U langchain-text-splitters faiss-cpu sentence-transformers rank_bm25 langgraph datasets accelerate

In [ ]:
# Load the helper modules. In Colab, clone the repo. Locally, run from this folder.
import os, sys
if not os.path.exists("rag_pipeline.py"):
    if not os.path.exists("LLM-Library-Minds"):
        !git clone -q https://github.com/Anvithanm/LLM-Library-Minds.git
    os.chdir("LLM-Library-Minds/RAG-llm-knowledge-integration")
sys.path.insert(0, os.getcwd())

import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 120)

from rag_pipeline import (RAGConfig, RAGPipeline, HFChatLLM, load_books, chunk_books, build_langgraph)
from rag_evaluation import (
    build_testset, generate_synthetic_testset, split_dev_test, IndexFactory, default_ablation_configs,
    compare_retrieval_configs, calibrate_abstention, evaluate_end_to_end, LLMJudge,
    comparison_table, qtype_table, save_results, show_failures, export_for_human_review, judge_human_agreement)

OUT_DIR = "eval_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
MAX_TEST_ITEMS = None   # e.g. 30 for a quick run

Mistral-7B-Instruct is a gated model. Accept its license on Hugging Face and add an `HF_TOKEN` secret in Colab (key icon in the left sidebar).

## 1) Load the books

> **Data-quality note:** some dataset titles don't match the text. `"The Mysterious Island"` (Anthony Hope) is not Verne's novel, `"The Life of Julius Caesar"` (Melville) is *Bartleby, the Scrivener*, and `"The Lord of the Rings"` (Tolkien) is Machiavelli's *The Prince*. Apostrophes are also stripped (`won t`). The system answers with the dataset's labels, so a question like *"Who is Frodo in The Lord of the Rings?"* will retrieve Machiavelli.

In [ ]:
books = load_books(n_books=20)
pd.DataFrame([{k: b[k] for k in ("book_id", "title", "author", "category")} | {"chars": len(b["text"])} for b in books])

## 2) Load the LLM

The same wrapper is used for answering, writing synthetic questions, and judging.
For less biased judging, create a second `HFChatLLM` with a different model (e.g. `Qwen/Qwen2.5-7B-Instruct` with `load_in_4bit=True`, after `pip install bitsandbytes`) and pass it to `LLMJudge`. A model grading its own answers tends to be too lenient.

In [ ]:
llm = HFChatLLM("mistralai/Mistral-7B-Instruct-v0.3")
print(llm([{"role": "user", "content": "In one sentence, what is retrieval-augmented generation?"}], max_new_tokens=60))

## 3) Build the test set

* **Curated** ([eval_data/curated_questions.json](eval_data/curated_questions.json)): hand-written questions whose evidence quotes were checked against the book text.
* **Synthetic:** chunks are sampled evenly across all 20 books (skipping front matter). The LLM writes a question, an answer, and an *exact* supporting quote. Items are dropped if the quote can't be found, if the question refers to "the passage", or if it duplicates another question. The quote's position in the book becomes the gold span. The set is cached in `eval_data/synthetic_questions.json`, so later runs reuse the same questions.
* **Negatives** ([eval_data/negative_questions.json](eval_data/negative_questions.json)): off-topic questions, books that are in the dataset but not indexed (Frankenstein, Monte Cristo...), false premises, and gibberish.

Add your own questions to the JSON files to grow the curated set. That makes the evaluation more reliable over time.

In [ ]:
chunks = chunk_books(books, chunk_size=1000, chunk_overlap=200)
synthetic = generate_synthetic_testset(books, chunks, llm, n_questions=60,
                                       out_path="eval_data/synthetic_questions.json")
items = build_testset(books, synthetic)
dev, test = split_dev_test(items, dev_frac=0.3, seed=0)
if MAX_TEST_ITEMS:  # random sample so every question type stays represented
    import random
    test = random.Random(0).sample(test, min(MAX_TEST_ITEMS, len(test)))
print(f"{len(items)} questions -> dev {len(dev)} / test {len(test)}")
pd.crosstab(pd.Series([i["qtype"] for i in items], name="qtype"),
            pd.Series(["dev" if i in dev else "test" for i in items], name="split"))

In [ ]:
# Spot-check a few synthetic questions. Their quality limits how much the scores can be trusted.
pd.DataFrame(synthetic)[["title", "question", "reference_answer"]].sample(8, random_state=0)

## 4) Retrieval evaluation and ablation

These scores need no LLM, so it's cheap to compare many configurations. Each row adds one change.

* **Hit@k:** share of answerable questions where a chunk containing the gold evidence is in the top k. The generator sees `top_k = 4` chunks, so Hit@3 to Hit@5 is the number that matters most.
* **MRR / nDCG@10:** how high the first relevant chunk is ranked.
* **book_hit@k:** a looser check: did the right *book* show up at all?
* **abstain_auroc:** probability that a random answerable question gets a higher retrieval score than a random unanswerable one. 1.0 means a single threshold separates them perfectly.

In [ ]:
factory = IndexFactory(books, cache_dir="emb_cache")   # shares models + embeddings across configs
ablation, ablation_items = compare_retrieval_configs(factory, default_ablation_configs(), test)
ablation[["hit@1", "hit@3", "hit@5", "hit@10", "book_hit@5", "mrr", "ndcg10", "abstain_auroc", "latency_p50_ms"]].round(3)

In [ ]:
ablation.to_csv(f"{OUT_DIR}/retrieval_ablation.csv")

## 5) Calibrate the abstention threshold (dev split only)

The system refuses to answer when its best retrieval score is below a threshold. We choose the threshold with the best balanced accuracy (share of answerable questions answered + share of unanswerable ones refused) on **dev**, while still answering at least 90% of answerable dev questions. Tuning on the test set would make the results look better than they really are.

In [ ]:
v1_index = factory.build(RAGConfig.baseline())   # MiniLM, dense, no headers/rerank/routing
v2_index = factory.build(RAGConfig())            # full v2

cal_v1 = calibrate_abstention(v1_index, dev)
cal_v2 = calibrate_abstention(v2_index, dev)
for name, cal in [("v1 (cosine)", cal_v1), ("v2 (reranker)", cal_v2)]:
    b = cal["best"]
    print(f"{name:14s} threshold={cal['threshold']:.3f}  dev AUROC={cal['auroc']:.2f}  "
          f"answers {b['answer_coverage']:.0%} of answerable, refuses {b['negative_refusal']:.0%} of unanswerable")

## 6) End-to-end evaluation on the test split

Three systems are compared:
1. **v1 retrieval:** v1's retriever with a fairly calibrated cosine gate (generation bug fixed, otherwise it can't be scored at all)
2. **v2 without gate:** v2 retrieval, relying only on the prompt to say "I don't know"
3. **v2:** v2 retrieval with the calibrated re-ranker gate

For each answerable question, the judge grades correctness against the reference (1-5; 4 or above counts as correct). It also splits the answer into individual claims and checks each against the retrieved context. Negatives count as handled when the system refuses, or, for false-premise questions, when it doesn't go along with the premise.

In [ ]:
judge = LLMJudge(llm)               # or LLMJudge(other_llm) for a different judge model
sim_model = factory._model("embed", "BAAI/bge-small-en-v1.5")

systems = {
    "v1 retrieval": RAGPipeline(v1_index, llm, RAGConfig.baseline(abstain_threshold=cal_v1["threshold"])),
    "v2 (no gate)": RAGPipeline(v2_index, llm, RAGConfig(abstain_threshold=None)),
    "v2":           RAGPipeline(v2_index, llm, RAGConfig(abstain_threshold=cal_v2["threshold"])),
}
summaries, item_results = {}, {}
for name, pipe in systems.items():
    print(f"\n===== {name} =====")
    summaries[name], item_results[name] = evaluate_end_to_end(pipe, test, judge=judge, embedder=sim_model)
    save_results(f"{OUT_DIR}/{name.replace(' ', '_').replace('(', '').replace(')', '')}", summaries[name], item_results[name])
print("judge JSON parse failures:", judge.parse_failures)

### Results

Each value is shown as `mean [95% bootstrap CI]`. If two systems' confidence intervals overlap a lot, the test set is too small to call the difference real. Add more curated or synthetic questions.

In [ ]:
comparison_table(summaries)

In [ ]:
# Success rate by question type (answerable: judged correct; negatives: handled correctly)
qtype_table(summaries).round(2)

In [ ]:
# Where do the failures come from? (retrieval miss vs. hallucination vs. generation error vs. refusal)
pd.DataFrame({name: s["outcomes"] for name, s in summaries.items()}).fillna(0).astype(int)

## 7) Failure analysis

In [ ]:
show_failures(item_results["v2"], "wrong", n=5)
show_failures(item_results["v2"], "false_refusal", n=3)
show_failures(item_results["v2"], "answered_unanswerable", n=3)

## 8) Validate the judge with a human spot check

LLM-judge scores are only useful if they agree with people. Label about 30 answers by hand in the exported CSV (`1` = correct, `0` = not), then compute agreement. A Cohen's kappa above about 0.6 means the judge's numbers are reasonable to report.

In [ ]:
export_for_human_review(item_results["v2"], f"{OUT_DIR}/human_review.csv", n=30)
# ...after filling in the last column:
# judge_human_agreement(f"{OUT_DIR}/human_review.csv")

## 9) Try it

In [ ]:
rag = systems["v2"]

def ask(question):
    out = rag.answer(question)
    print(f"Q: {question}\nA: {out['answer']}\n")
    if not out["abstained_by_gate"]:
        for i, h in enumerate(out["hits"], 1):
            print(f"  [{i}] {h.chunk.title} (chars {h.chunk.start}-{h.chunk.end}) rerank={h.rerank:.2f}")
    print(f"  gate score={out['gate_score']:.3f}  latency={out['timings']['total_s']:.1f}s")

ask("Which book has the character Robinson Kreutznaer?")
ask("What does the parrot keep screaming in Treasure Island?")
ask("What is the capital of France?")

In [ ]:
# The same pipeline as a LangGraph graph (retrieve -> gate -> generate | abstain)
graph = build_langgraph(rag)
graph.invoke({"question": "Who painted the portrait of Dorian Gray?"})["answer"]

## Caveats

* **The judge grading its own model:** Mistral judging Mistral tends to be lenient. Use a different judge model and check against human labels (section 8).
* **Synthetic questions** come from single chunks, so they favor "find one passage" lookups. The curated set and the per-type table show how the system does on cross-book, metadata ("which books are by Dickens?"), and false-premise questions. Those are harder and closer to what real users ask.
* **Sample size:** with about 70 test questions, a 5-point difference is usually inside the confidence interval. Treat small differences as noise.